In [ ]:
# ============================================================================#          FRAUD DETECTION - COMPLETE MODEL COMPARISON#          Comparing: Logistic Regression, Random Forest, SVM, XGBoost, Naive Bayes#          Baseline: ANN (Sensitivity 81.78%, Precision 70.18%, F1 75.49%)# ============================================================================println("🚀 E-Commerce Fraud Detection - Model Comparison Analysis")println("="^70)using Pkgprintln("\n📦 Installing required packages (this may take several minutes)...")packages = ["CSV", "DataFrames", "Statistics", "Random", "Dates", "StatsBase",            "MLJ", "MLJLinearModels", "MLJDecisionTreeInterface",             "MLJLIBSVMInterface", "XGBoost", "Plots", "StatsPlots"]for pkg in packages    try        Pkg.add(pkg)    catch e        println("  ⚠️  Could not install $pkg")    endendprintln("\n📚 Loading packages...")using CSV, DataFrames, Statistics, Random, Dates, StatsBaseusing MLJusing Plots, StatsPlotsprintln("✅ Packages loaded successfully!")

In [ ]:
# ============================================================================#              LOAD UTILITY FUNCTIONS# ============================================================================# Load existing utility functions from utils.jlinclude("utils.jl")# Load model comparison specific utilitiesinclude("model_utils.jl")println("✅ Utility functions loaded!")

In [ ]:
# ============================================================================#              DATA LOADING & PREPROCESSING# ============================================================================# Same preprocessing function as miglior compromesso.ipynbfunction preprocess_data_enhanced(dataframe)    data = copy(dataframe)        println("\n🔧 Creating Enhanced Features...")        # TIME FEATURES    if "Transaction Date" in names(data)        try            if eltype(data[!, "Transaction Date"]) <: AbstractString                 data.ParsedDate = DateTime.(data[!, "Transaction Date"], dateformat"y-m-d H:M:S")             else                 data.ParsedDate = data[!, "Transaction Date"]            end            data.Hour = hour.(data.ParsedDate)            data.Is_Night = [h < 6 ? 1.0 : 0.0 for h in data.Hour]            data.Is_Weekend = [dayofweek(d) in [6,7] ? 1.0 : 0.0 for d in data.ParsedDate]            data.Hour_Risk = [h in [0,1,2,3,4,5,23] ? 1.0 : 0.0 for h in data.Hour]            data.Is_Early_Morning = [h in [2,3,4,5] ? 1.0 : 0.0 for h in data.Hour]            println("  ✓ Time features created")        catch e            println("  ⚠ Error: $e")        end    end    # IMPUTATION    for col in ["Transaction Amount", "Quantity", "Customer Age", "Account Age Days"]        if col in names(data) && any(ismissing, data[!, col])            median_val = median(skipmissing(data[!, col]))            replace!(data[!, col], missing => median_val)        end    end        # RISK FEATURES    if "Transaction Amount" in names(data) && "Account Age Days" in names(data)        data.Amount_per_AccountAge = data[!, "Transaction Amount"] ./ (data[!, "Account Age Days"] .+ 1.0)    end        if "Transaction Amount" in names(data)        p95 = quantile(data[!, "Transaction Amount"], 0.95)        p99 = quantile(data[!, "Transaction Amount"], 0.99)        data.High_Value_Flag = [amt > p95 ? 1.0 : 0.0 for amt in data[!, "Transaction Amount"]]        data.Very_High_Value_Flag = [amt > p99 ? 1.0 : 0.0 for amt in data[!, "Transaction Amount"]]    end        if "Quantity" in names(data)        data.High_Qty_Flag = [q > 5 ? 1.0 : 0.0 for q in data[!, "Quantity"]]        data.Very_High_Qty_Flag = [q > 10 ? 1.0 : 0.0 for q in data[!, "Quantity"]]                if "Transaction Amount" in names(data)            data.Unit_Price = data[!, "Transaction Amount"] ./ (data[!, "Quantity"] .+ 0.1)            p95_unit = quantile(data.Unit_Price, 0.95)            data.High_Unit_Price_Flag = [up > p95_unit ? 1.0 : 0.0 for up in data.Unit_Price]        end    end        if "Account Age Days" in names(data)        data.New_Account_Flag = [age < 30 ? 1.0 : 0.0 for age in data[!, "Account Age Days"]]        data.Very_New_Account_Flag = [age < 7 ? 1.0 : 0.0 for age in data[!, "Account Age Days"]]    end        if "Customer Age" in names(data)        data.Young_Customer_Flag = [age < 25 ? 1.0 : 0.0 for age in data[!, "Customer Age"]]        data.Senior_Customer_Flag = [age > 65 ? 1.0 : 0.0 for age in data[!, "Customer Age"]]    end        # COMBINED RISK SCORE    risk_cols = []    for col in ["High_Value_Flag", "High_Qty_Flag", "New_Account_Flag",                 "Hour_Risk", "Is_Night", "Young_Customer_Flag"]        if col in names(data)            push!(risk_cols, col)        end    end        if !isempty(risk_cols)        data.Risk_Score = sum([data[!, col] for col in risk_cols])    end        # CLEANUP    cols_to_drop = ["Transaction ID", "Customer ID", "Transaction Date", "ParsedDate",                     "IP Address", "Shipping Address", "Billing Address", "Customer Location"]    select!(data, Not(intersect(names(data), cols_to_drop)))        # ONE-HOT ENCODING    categorical_cols = ["Payment Method", "Product Category", "Device Used"]    input_df = data[:, setdiff(names(data), categorical_cols)]        for col in categorical_cols        if col in names(data)            encoded_matrix = oneHotEncoding(data[!, col])            new_col_names = ["$(col)_$(i)" for i in 1:size(encoded_matrix, 2)]            encoded_df = DataFrame(encoded_matrix, new_col_names)            input_df = hcat(input_df, encoded_df)        end    end        println("✅ Feature engineering completed!")    return input_dfend# LOAD DATAconst DATA_PATH = "Fraudulent_E-Commerce_Transaction_Data_merge.csv"if !isfile(DATA_PATH)    error("Dataset not found! Please download from Kaggle and save as: $DATA_PATH")endprintln("\n📂 Loading dataset...")df = CSV.read(DATA_PATH, DataFrame)target_col = "Is Fraudulent"# Balance 50/50fraud_rows = df[df[:, target_col] .== 1, :]n_fraud = size(fraud_rows, 1)println("   Total frauds: $n_fraud")non_fraud_rows = df[df[:, target_col] .== 0, :]non_fraud_sample = non_fraud_rows[shuffle(1:size(non_fraud_rows, 1))[1:n_fraud], :]df_balanced = vcat(fraud_rows, non_fraud_sample)df_balanced = df_balanced[shuffle(1:size(df_balanced, 1)), :] println("   Balanced dataset: $(size(df_balanced, 1)) samples")# Preprocessdf_processed = preprocess_data_enhanced(df_balanced)# Prepare arraysinput_cols = setdiff(names(df_processed), [target_col])inputs = Matrix{Float64}(df_processed[:, input_cols])targets_bool = Bool.(vec(df_processed[:, target_col]))println("\n📊 Dataset Info:")println("   Features: $(size(inputs, 2))")println("   Samples: $(size(inputs, 1))")# Create 3-fold CV indicesRandom.seed!(42)cv_indices = crossvalidation(targets_bool, 3)println("\n✅ Data ready for modeling!")

In [ ]:
# ============================================================================#              MODEL 1: LOGISTIC REGRESSION# ============================================================================println("\n🔄 Training Logistic Regression with 3-fold CV...")LogisticClassifier = @load LogisticClassifier pkg=MLJLinearModelslr_metrics = []for fold in 1:3    println("  Fold $fold/3...")        test_mask = cv_indices .== fold    train_mask = .!test_mask        X_train = inputs[train_mask, :]    y_train = targets_bool[train_mask]    X_test = inputs[test_mask, :]    y_test = targets_bool[test_mask]        # Normalize (required for LogReg)    norm_params = calculateMinMaxNormalizationParameters(X_train)    X_train_norm = normalizeMinMax(X_train, norm_params)    X_test_norm = normalizeMinMax(X_test, norm_params)        # Tune lambda (L2 regularization)    best_sens = 0.0    best_pred = nothing        for lambda in [0.001, 0.01, 0.1, 1.0, 10.0]        model = LogisticClassifier(lambda=lambda)        mach = machine(model, X_train_norm, y_train)        MLJ.fit!(mach, verbosity=0)                y_pred = MLJ.predict_mode(mach, X_test_norm)        m = calculate_comprehensive_metrics(y_test, y_pred)                if m["sensitivity"] > best_sens            best_sens = m["sensitivity"]            best_pred = y_pred        end    end        push!(lr_metrics, calculate_comprehensive_metrics(y_test, best_pred))endlr_avg = aggregate_cv_metrics(lr_metrics)print_model_results("LOGISTIC REGRESSION", lr_avg)

In [ ]:
# ============================================================================#              MODEL 2: RANDOM FOREST# ============================================================================println("\n🔄 Training Random Forest with 3-fold CV...")RandomForestClassifier = @load RandomForestClassifier pkg=DecisionTreerf_metrics = []for fold in 1:3    println("  Fold $fold/3...")        test_mask = cv_indices .== fold    train_mask = .!test_mask        X_train = inputs[train_mask, :]    y_train = targets_bool[train_mask]    X_test = inputs[test_mask, :]    y_test = targets_bool[test_mask]        best_sens = 0.0    best_pred = nothing        # Hyperparameter tuning    for n_trees in [100, 200]        for max_d in [10, 20, -1]            model = RandomForestClassifier(n_trees=n_trees, max_depth=max_d)            mach = machine(model, X_train, y_train)            MLJ.fit!(mach, verbosity=0)                        y_pred = MLJ.predict_mode(mach, X_test)            m = calculate_comprehensive_metrics(y_test, y_pred)                        if m["sensitivity"] > best_sens                best_sens = m["sensitivity"]                best_pred = y_pred            end        end    end        push!(rf_metrics, calculate_comprehensive_metrics(y_test, best_pred))endrf_avg = aggregate_cv_metrics(rf_metrics)print_model_results("RANDOM FOREST", rf_avg)

In [ ]:
# ============================================================================#              MODEL 3: SUPPORT VECTOR MACHINE (SVM)# ============================================================================println("\n🔄 Training SVM with 3-fold CV...")SVC = @load SVC pkg=LIBSVMsvm_metrics = []for fold in 1:3    println("  Fold $fold/3...")        test_mask = cv_indices .== fold    train_mask = .!test_mask        X_train = inputs[train_mask, :]    y_train = targets_bool[train_mask]    X_test = inputs[test_mask, :]    y_test = targets_bool[test_mask]        # SVM requires normalization    norm_params = calculateMinMaxNormalizationParameters(X_train)    X_train_norm = normalizeMinMax(X_train, norm_params)    X_test_norm = normalizeMinMax(X_test, norm_params)        best_sens = 0.0    best_pred = nothing        for cost in [0.1, 1.0, 10.0]        try            model = SVC(kernel="rbf", cost=cost)            mach = machine(model, X_train_norm, y_train)            MLJ.fit!(mach, verbosity=0)                        y_pred = MLJ.predict_mode(mach, X_test_norm)            m = calculate_comprehensive_metrics(y_test, y_pred)                        if m["sensitivity"] > best_sens                best_sens = m["sensitivity"]                best_pred = y_pred            end        catch e            println("    ⚠️  SVM failed: $e")        end    end        if best_pred !== nothing        push!(svm_metrics, calculate_comprehensive_metrics(y_test, best_pred))    endendsvm_avg = !isempty(svm_metrics) ? aggregate_cv_metrics(svm_metrics) : nothingif svm_avg !== nothing    print_model_results("SUPPORT VECTOR MACHINE", svm_avg)else    println("⚠️  SVM training failed across all folds")end

In [ ]:
# ============================================================================#              MODEL 4: GRADIENT BOOSTING (XGBoost)# ============================================================================println("\n🔄 Training XGBoost with 3-fold CV...")using XGBoostXGBoostClassifier = @load XGBoostClassifier pkg=XGBoostxgb_metrics = []for fold in 1:3    println("  Fold $fold/3...")        test_mask = cv_indices .== fold    train_mask = .!test_mask        X_train = inputs[train_mask, :]    y_train = targets_bool[train_mask]    X_test = inputs[test_mask, :]    y_test = targets_bool[test_mask]        best_sens = 0.0    best_pred = nothing        for eta in [0.01, 0.05, 0.1]        for depth in [3, 5, 7]            for rounds in [100, 200]                try                    model = XGBoostClassifier(                        eta=eta,                        max_depth=depth,                        num_round=rounds,                        objective="binary:logistic"                    )                    mach = machine(model, X_train, y_train)                    MLJ.fit!(mach, verbosity=0)                                        y_pred = MLJ.predict_mode(mach, X_test)                    m = calculate_comprehensive_metrics(y_test, y_pred)                                        if m["sensitivity"] > best_sens                        best_sens = m["sensitivity"]                        best_pred = y_pred                    end                catch e                    # Silently continue                end            end        end    end        if best_pred !== nothing        push!(xgb_metrics, calculate_comprehensive_metrics(y_test, best_pred))    endendxgb_avg = !isempty(xgb_metrics) ? aggregate_cv_metrics(xgb_metrics) : nothingif xgb_avg !== nothing    print_model_results("XGBOOST", xgb_avg)else    println("⚠️  XGBoost training failed")end

In [ ]:
# ============================================================================#              MODEL 5: NAIVE BAYES (GaussianNB)# ============================================================================println("\n🔄 Training Naive Bayes with 3-fold CV...")try    using NaiveBayes        nb_metrics = []        for fold in 1:3        println("  Fold $fold/3...")                test_mask = cv_indices .== fold        train_mask = .!test_mask                X_train = inputs[train_mask, :]'  # NaiveBayes expects features x samples        y_train = targets_bool[train_mask]        X_test = inputs[test_mask, :]'        y_test = targets_bool[test_mask]                model = GaussianNB(unique(y_train))        NaiveBayes.fit(model, X_train, y_train)                y_pred = NaiveBayes.predict(model, X_test)                push!(nb_metrics, calculate_comprehensive_metrics(y_test, y_pred))    end        nb_avg = aggregate_cv_metrics(nb_metrics)    print_model_results("NAIVE BAYES", nb_avg)catch e    println("⚠️  Naive Bayes not available: $e")    nb_avg = nothingend

In [ ]:
# ============================================================================#              FINAL COMPARISON & ANALYSIS# ============================================================================println("\n\n" * "="^80)println("📊 COMPLETE MODEL COMPARISON")println("="^80)# Baseline ANN from miglior compromesso.ipynbann_baseline = Dict(    "accuracy" => 0.7341,    "sensitivity" => 0.8178,    "specificity" => 0.6504,    "precision" => 0.7018,    "f1" => 0.7549,    "f2" => 0.7826,    "TP" => 61385,    "TN" => 48766,    "FP" => 26241,    "FN" => 13675)# Collect all modelsall_models = Dict("ANN (Baseline)" => ann_baseline)if @isdefined(lr_avg)    all_models["Logistic Regression"] = lr_avgendif @isdefined(rf_avg)    all_models["Random Forest"] = rf_avgendif @isdefined(svm_avg) && svm_avg !== nothing    all_models["SVM"] = svm_avgendif @isdefined(xgb_avg) && xgb_avg !== nothing    all_models["XGBoost"] = xgb_avgendif @isdefined(nb_avg) && nb_avg !== nothing    all_models["Naive Bayes"] = nb_avgend# Create comparison tablecreate_comparison_table(all_models)# Business impact analysisprint_business_analysis(all_models)# Find best model by sensitivitybest_model_name = ""best_sensitivity = 0.0for (name, metrics) in all_models    if metrics["sensitivity"] > best_sensitivity        best_sensitivity = metrics["sensitivity"]        best_model_name = name    endendprintln("\n" * "="^80)println("🏆 RECOMMENDATIONS")println("="^80)println("\n1. BEST FOR PRODUCTION (Highest Sensitivity):")println("   ⭐ $best_model_name")println("   Sensitivity: $(round(best_sensitivity*100, digits=2))%")println("   This model catches the most frauds!")println("\n2. TRADE-OFFS:")println("   - Sensitivity ↑ → More frauds detected")println("   - Sensitivity ↑ → More false positives (review cost)")println("   - Balance depends on business priorities")println("\n3. NEXT STEPS:")println("   ✓ Deploy best model with A/B testing")println("   ✓ Monitor precision/recall in production")println("   ✓ Consider ensemble methods")println("   ✓ Retrain monthly with new data")println("\n" * "="^80)println("✅ MODEL COMPARISON COMPLETE!")println("="^80)

In [ ]:
# ============================================================================#              VISUALIZATIONS# ============================================================================println("\n📊 Creating visualizations...")# Prepare datamodel_names = collect(keys(all_models))sensitivities = [all_models[m]["sensitivity"] for m in model_names]precisions = [all_models[m]["precision"] for m in model_names]f1_scores = [all_models[m]["f1"] for m in model_names]# Create comparison bar chartusing Plotsp = groupedbar(    [sensitivities precisions f1_scores] .* 100,    bar_position=:dodge,    bar_width=0.7,    xticks=(1:length(model_names), model_names),    xlabel="Model",    ylabel="Score (%)",    title="Model Comparison - Key Metrics",    label=["Sensitivity" "Precision" "F1 Score"],    legend=:best,    ylims=(0, 100),    size=(800, 500),    xrotation=45)display(p)println("\n✅ Analysis complete!")